# Agent 3 — Cognitive / Fatigue Prediction

NASA Space Apps · PMData research prototype · Cognitive v1.0.0

This notebook documents the completed Agent 3 workflow: next-day elevated self-reported fatigue (scores 1–2), using historical inputs through calendar day t−1. It uses the reusable `ml.cognitive` modules and the approved Agent 2 preprocessing v1.0.0. Average precision (AP) is the primary selection metric; E0–E5 and the chronological and grouped evaluation protocols follow the frozen ML specification.

**Use private Kaggle inputs and a private notebook.** This GitHub-ready copy contains no saved execution outputs or participant-level data. It preserves the model-selection, evaluation, and export code; the historical aggregate results below are from the original executed notebook, not a new run.

### Execution guide
1. Mount the private code, preprocessed data, research and report inputs. Run setup, diagnostics and validation.
2. Review preprocessing evidence; deliberately approve development. Start with FAST mode if needed, then FULL mode for the official run. Save the fast comparison before changing modes.
3. Review the validation leaderboard, baselines, calibration and threshold analysis.
4. Deliberately approve the frozen configuration and evaluate the protected chronological test, grouped procedure and sensitivity scenarios. The CatBoost correction is isolated from the original selected checkpoint.
5. Review evaluation results; deliberately approve export. Download the model and reports before the Kaggle session ends.

By default, only contract validation/diagnostics run. Do not enable approval flags for new data without actual ML lead review. Do not use protected test results to select or tune models. No clinical diagnosis, objective cognitive impairment prediction or astronaut clearance is implied.

## 1. Private input mounts and environment setup

Attach `nasa-agent3-code`, the Agent 2 data dataset (the completed run used `pmdata-preprocessed1`), `pmdata-research`, and `pmdata-reports` to Kaggle. The code root must contain `ml/cognitive/`. Use environment variables to override mount paths; otherwise the helper discovers the unique attached code dataset. A local complete Agent 2 deliverable can instead be supplied with `PMDATA_PREPROCESSED_ROOT`.

The setup reconstructs the expected directory layout under `/kaggle/working`, without modifying the mounted inputs. Re-run this cell after a session reset.

In [ ]:
from pathlib import Path
from dataclasses import replace
import os
import sys
import json
import shutil

KAGGLE_INPUT = Path("/kaggle/input")

def find_code_root():
    override = os.environ.get("COGNITIVE_CODE_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    if (Path.cwd() / "ml/cognitive").is_dir():
        return Path.cwd().resolve()
    matches = [
        p.resolve()
        for p in KAGGLE_INPUT.rglob("nasa-agent3-code")
        if (p / "ml/cognitive").is_dir()
    ]
    if len(set(matches)) != 1:
        raise RuntimeError(
            "Set COGNITIVE_CODE_ROOT to the mounted repository root "
            "containing ml/cognitive."
        )
    return matches[0]

CODE_ROOT = find_code_root()
if not (CODE_ROOT / "ml/cognitive").is_dir():
    raise RuntimeError("CODE_ROOT must contain ml/cognitive.")

OUTPUT_ROOT = Path(
    os.environ.get("COGNITIVE_OUTPUT_ROOT", "/kaggle/working/cognitive_outputs")
).resolve()

# Use an already assembled deliverable if explicitly provided.
upstream_override = os.environ.get("PMDATA_PREPROCESSED_ROOT")
if upstream_override:
    UPSTREAM_ROOT = Path(upstream_override).expanduser().resolve()
else:
    BASE = Path(
        os.environ.get("PMDATA_KAGGLE_BASE", str(CODE_ROOT.parent))
    ).resolve()
    WORK = Path("/kaggle/working/pmdata_preprocessing_ready")

    # The completed Kaggle run used pmdata-preprocessed1.
    data_slug = os.environ.get("PMDATA_DATA_SLUG", "pmdata-preprocessed1")
    data_mount = BASE / data_slug
    if not data_mount.is_dir() and "PMDATA_DATA_SLUG" not in os.environ:
        data_mount = BASE / "pmdata-preprocessed"
    data_src = data_mount / "data/processed"
    research_mount = BASE / "pmdata-research"
    reports_mount = BASE / "pmdata-reports"

    def unique_parent(root, filename):
        matches = list(root.rglob(filename))
        if len(matches) != 1:
            raise RuntimeError(f"Expected exactly one {filename} under {root}.")
        return matches[0].parent

    research_src = unique_parent(research_mount, "feature_schema.csv")
    reports_src = unique_parent(reports_mount, "preprocessing_report.md")

    # Reconstruct the layout required by the original loader.
    shutil.copytree(data_src, WORK / "data/processed", dirs_exist_ok=True)
    shutil.copytree(research_src, WORK / "research", dirs_exist_ok=True)
    shutil.copytree(reports_src, WORK / "reports", dirs_exist_ok=True)
    UPSTREAM_ROOT = WORK.resolve()

required_files = [
    "data/processed/fatigue_dataset.parquet",
    "data/processed/split_manifest.csv",
    "data/processed/grouped_split_manifest.csv",
    "data/processed/preprocessing_run.json",
    "research/feature_schema.csv",
    "reports/preprocessing_report.md",
    "reports/leakage_test_report.md",
    "reports/delivery_integrity.json",
]
missing = [name for name in required_files if not (UPSTREAM_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError("Missing required preprocessing files: " + ", ".join(missing))

sys.path.insert(0, str(CODE_ROOT))
from ml.cognitive.config import Config
from ml.cognitive.train import execute
from ml.cognitive.data_loader import load_and_validate
from ml.cognitive.models import gpu_inventory

print("Code root:", CODE_ROOT)
print("Upstream root:", UPSTREAM_ROOT)
print("Required files:", len(required_files), "present")


## 2. Execution settings — safe defaults

FULL mode is the default experiment budget (250 initial trees/iterations; 30 Optuna trials per top configuration). FAST mode keeps the same model/feature matrix with reduced budgets (30 initial trees/iterations, at most 5 tuning trials). Changing the configuration requires a new development approval. The grouped protocol has its own fixed inner budget in the frozen configuration.

All approval switches start disabled. Enable them only after reviewing the relevant evidence; set the reviewer and merge reference before development. `RUN_*` for protected stages are enabled by the corresponding approval cell, not by a default Run All. Keep the existing full-run configuration when using a previously frozen model.

In [ ]:
SEED = 42
FAST_MODE = False
REQUESTED_DEVICE = "auto"
RUN_DEVELOPMENT = False
RUN_TUNING = True
RUN_SUPPLEMENTARY = True
RUN_SENSITIVITY_VALIDATION = True
RUN_FINAL_TEST = False
RUN_GROUPED_OUTER = False
RUN_SENSITIVITY_TEST = False
RUN_EXPORT = False
RUN_GPU_SOFTWARE_SMOKE = False
# Explicit human decisions; never infer approval from a successful validation.
APPROVE_DEVELOPMENT = False
APPROVE_FROZEN_EVALUATION = False
APPROVE_GROUPED_BUGFIX = False
APPROVE_EXPORT = False
APPROVED_BY = ""  # Enter the actual ML lead reviewer.
REVIEW_CONFIRMED = False
PREPROCESSING_MERGED_TO_MAIN = False
MERGE_REFERENCE = ""  # Enter the verified PR / commit reference.

APPROVAL_RECORD = None
FROZEN_CONFIG = None    # Approved configs/frozen_config.json from completed development.
RESUME_RUN = None       # Exact matching private run directory, optional.
config = Config(
    data_dir=str(UPSTREAM_ROOT / "data/processed"),
    schema_path=str(UPSTREAM_ROOT / "research/feature_schema.csv"),
    reports_root=str(UPSTREAM_ROOT / "reports"),
    output_root=str(OUTPUT_ROOT),
    mode="validate",
    seed=SEED,
    fast_mode=FAST_MODE,
    requested_device=REQUESTED_DEVICE,
    approval_record=APPROVAL_RECORD,
    final_config=FROZEN_CONFIG,
    resume_run=RESUME_RUN,
    run_tuning=RUN_TUNING,
    run_supplementary=RUN_SUPPLEMENTARY,
    run_sensitivity_validation=RUN_SENSITIVITY_VALIDATION,
    gpu_smoke=RUN_GPU_SOFTWARE_SMOKE,
)
print("Validation-only defaults; no training approval inferred.")


## 3. Environment and GPU inventory

Logistic Regression and Random Forest remain CPU. XGBoost uses modern hist + cuda, CatBoost GPU + devices=0, and LightGBM requires a supported OpenCL/CUDA build. Driver inventory is not successful library GPU verification. An optional explicit synthetic GPU fit+predict smoke is configured separately; hardware/library failures and CPU retries are always logged.

In [ ]:
print(json.dumps(gpu_inventory(), indent=2))
import importlib.metadata
for package in ("numpy", "pandas", "scikit-learn", "xgboost", "lightgbm", "catboost", "optuna", "pyarrow"):
    try: print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError: print(package, "MISSING — relevant stage blocked/skipped explicitly")


## 4. Read-only contract and preprocessing gate

The loader verifies saved hashes, allowlist, labels, exact calendar alignment, purges, masks and group integrity. It exposes training/validation support, never protected test outcome distributions. A successful software gate is not ML-lead approval and does not infer the required preprocessing merge.

In [ ]:
validation_run, validation_result = execute(config)
print("Validation artifact directory:", validation_run)


## 5. Preprocessing evidence and development approval

Review both reports and confirm the shared preprocessing version is merged to `main`. The data-contract check is not itself human approval. A new run requires the ML lead to confirm audit reconciliation, timestamp assumptions and the shared feature contract. Actual ingestion and Fitbit timezone remain unresolved.

Set `APPROVE_DEVELOPMENT`, `REVIEW_CONFIRMED`, `PREPROCESSING_MERGED_TO_MAIN`, `APPROVED_BY` and `MERGE_REFERENCE` in section 2, then run this approval cell. A previous completed run used PR #1, merge commit `5df431d`; verify the reference for the data/code version actually used.

In [ ]:
from IPython.display import display, Markdown

display(Markdown(
    (UPSTREAM_ROOT / "reports/preprocessing_report.md").read_text()
))

display(Markdown(
    (UPSTREAM_ROOT / "reports/leakage_test_report.md").read_text()
))


In [ ]:
if APPROVE_DEVELOPMENT:
    from datetime import datetime, timezone
    from dataclasses import replace
    from ml.cognitive.approval import require_approval
    import json

    # Explicit ML lead approval; values are set in section 2.
    assert REVIEW_CONFIRMED is True, "Review preprocessing reports first."
    assert APPROVED_BY.strip(), "Enter the actual reviewer."
    assert PREPROCESSING_MERGED_TO_MAIN is True, "Confirm preprocessing merge."
    assert MERGE_REFERENCE.strip(), "Enter the verified PR / commit reference."

    RUN_GPU_SOFTWARE_SMOKE = False

    config = replace(
        config,
        fast_mode=FAST_MODE,
        run_tuning=RUN_TUNING,
        run_supplementary=RUN_SUPPLEMENTARY,
        run_sensitivity_validation=RUN_SENSITIVITY_VALIDATION,
        gpu_smoke=False,
        requested_device="auto"
    )

    # Validate current dataset and fingerprints
    data = load_and_validate(config)

    # Create approval record
    approval = {
        "approved": True,
        "task": "fatigue",
        "approved_by": APPROVED_BY,
        "approved_at_utc": datetime.now(timezone.utc).isoformat(),

        "preprocessing_reviewed": True,
        "preprocessing_merged_to_main": PREPROCESSING_MERGED_TO_MAIN,
        "preprocessing_version": data.metadata["version"],
        "gate_status": "CONDITIONAL GO",

        **data.fingerprints,

        "config_fingerprint": config.config_fingerprint,
        "authorized_scopes": ["development"],

        "approved_freeze_fingerprint": None,
        "calibration_adequacy_approved": False,

        "reviewed_evidence": {
            "mandatory_validation": True,
            "audit_reconciliation": True,
            "timestamp_assumptions": True,
            "shared_contract": True
        },

        "scenario_fingerprints": {
            scenario: load_and_validate(
                config, scenario
            ).fingerprints
            for scenario in (
                ["primary", "offset_plus1", "offset_plus2", "wellness_lag2"]
                if RUN_SENSITIVITY_VALIDATION else ["primary"]
            )
        },

        "merge_reference": MERGE_REFERENCE
    }

    approval_path = Path(
        "/kaggle/working/agent3_approval.json"
    )

    approval_path.write_text(
        json.dumps(approval, indent=2),
        encoding="utf-8"
    )

    APPROVAL_RECORD = str(approval_path)

    config = replace(
        config,
        approval_record=APPROVAL_RECORD
    )

    # Validate the approval record
    require_approval(config, data, scope="development")

    RUN_DEVELOPMENT = True

    print("ML lead approval validated successfully!")
    print("Approval record:", APPROVAL_RECORD)
    print("Configuration fingerprint:", config.config_fingerprint)
    print("Development training:", RUN_DEVELOPMENT)
else:
    print('BLOCKED: explicit development approval required.')


## 6. Authorized development: baselines, E0–E5, tuning, calibration and freeze

One orchestration function runs reusable stages with visible per-experiment/trial/fold UTC timestamps, measured timings, current leaderboards and incrementally saved results. Five required families × six groups = 30 initial configurations on identical rows. Baselines include training prevalence/majority, exact-lag persistence and approved past-only 28-day personal prevalence (min14); unavailable history blocks that baseline only. Ranking and Optuna optimize sklearn average precision. Calibration uses training-only forward OOF scores; validation adequacy and threshold halves are disjoint. Unsupported calibration yields risk score, not verified probability. Private CPU-loadable checkpoints support exact resume and bind the selected train-fitted weights to the freeze.

In [ ]:
development_run = None
if RUN_DEVELOPMENT:
    development_run, development_result = execute(replace(config, mode="develop"))
    print(json.dumps(development_result, indent=2))
else:
    print("BLOCKED: development/training/tuning are disabled; explicit ML-lead approval required.")


## 7. Validation review and result tables

Read the saved experiment results rather than parsing long training logs. The leaderboard is ordered by AP. Baselines, calibration summary, the selected configuration and figures are available after development. Review the official E0–E5 comparisons on identical rows and the optional supplementary/sensitivity runs separately.

The following cell creates the same review ZIP as the original workflow. Keep exported files private until their contents have been reviewed.

In [ ]:
if development_run is not None:
    import pandas as pd
    import json
    import zipfile
    from pathlib import Path
    from IPython.display import display, FileLink

    # Output directory
    export_dir = Path("/kaggle/working/agent3_review") / Path(development_run).name
    export_dir.mkdir(parents=True, exist_ok=True)

    metrics_dir = development_run / "metrics"

    # ==================================================
    # 1. Model leaderboard
    # ==================================================

    results = pd.read_csv(
        metrics_dir / "model_feature_results.csv"
    )

    columns = [
        "model_name",
        "feature_set",
        "source",
        "average_precision",
        "roc_auc",
        "macro_f1",
        "balanced_accuracy",
        "recall",
        "specificity",
        "brier",
        "elapsed_seconds",
        "actual_device",
        "n_features",
        "status"
    ]

    leaderboard_export = (
        results[columns]
        .sort_values("average_precision", ascending=False)
        .reset_index(drop=True)
    )

    print("\nMODEL LEADERBOARD")

    display(
        leaderboard_export.style.format(precision=4).background_gradient(
            subset=["average_precision", "roc_auc", "macro_f1", "recall"],
            cmap="Greens"
        )
    )

    leaderboard_export.to_csv(
        export_dir / "model_leaderboard.csv",
        index=False
    )

    # ==================================================
    # 2. Baseline comparison
    # ==================================================

    baseline_names = [
        "primary_validation_baseline_metrics.csv",
        "offset_plus1_sensitivity_validation_baseline_metrics.csv",
        "offset_plus2_sensitivity_validation_baseline_metrics.csv",
        "wellness_lag2_sensitivity_validation_baseline_metrics.csv",
    ]
    baseline_files = [metrics_dir / name for name in baseline_names
                      if (metrics_dir / name).is_file()]

    private_columns = {
        "participant_id", "participant",
        "subject_id", "subject",
        "cutoff_date", "cutoff_datetime",
        "target_fatigue_score"
    }

    for file in baseline_files:
        baseline_df = pd.read_csv(file)

        # Avoid exporting participant-level records
        if private_columns.intersection(baseline_df.columns):
            print(f"Skipped private report: {file.name}")
            continue

        if "participant" in file.name.lower():
            print(f"Skipped private report: {file.name}")
            continue

        print(f"\nBASELINE RESULTS: {file.name}")
        display(baseline_df)

        baseline_df.to_csv(
            export_dir / file.name,
            index=False
        )

    if not baseline_files:
        print("No baseline CSV found.")

    # ==================================================
    # 3. Calibration summary
    # ==================================================

    calibration_results = json.loads(
        (metrics_dir / "calibration_metrics.json")
        .read_text()
    )

    calibration_table = (
        pd.json_normalize(
            calibration_results,
            sep="."
        )
        .T.reset_index()
    )

    calibration_table.columns = [
        "Metric", "Value"
    ]

    print("\nCALIBRATION SUMMARY")
    display(calibration_table)

    calibration_table.to_csv(
        export_dir / "calibration_summary.csv",
        index=False
    )

    # ==================================================
    # 4. Selected model configuration
    # ==================================================

    frozen = json.loads(
        (
            development_run
            / "configs/frozen_config.json"
        ).read_text()
    )

    selected_model = {
        "model": frozen["model_name"],
        "feature_set": frozen["feature_set"],
        "n_features": len(frozen["features"]),
        "threshold": frozen["calibration"]["threshold"],
        "calibration_method": frozen["calibration"]["method"],
        "probability_verified": frozen["calibration"]["probability_verified"],
        **frozen["hyperparameters"]
    }

    selected_model_table = pd.DataFrame(
        list(selected_model.items()),
        columns=["Parameter", "Value"]
    )

    print("\nSELECTED MODEL CONFIGURATION")
    display(selected_model_table)

    selected_model_table.to_csv(
        export_dir / "selected_model.csv",
        index=False
    )

    # ==================================================
    # 5. Create downloadable ZIP
    # ==================================================

    zip_path = Path(
        "/kaggle/working/agent3_review_tables.zip"
    )

    with zipfile.ZipFile(
        zip_path, "w", zipfile.ZIP_DEFLATED
    ) as z:
        for file in export_dir.glob("*.csv"):
            z.write(file, arcname=file.name)

    print("\nExport completed!")
    print("ZIP location:", zip_path)

    display(
        FileLink("agent3_review_tables.zip")
    )
else:
    print('No development results to display.')


In [ ]:
if development_run is not None:
    import pandas as pd
    from IPython.display import display, Image
    for filename in ("runtime_summary.csv", "supplementary_results.csv", "sensitivity_results.csv"):
        path = development_run / "metrics" / filename
        if path.exists(): display(pd.read_csv(path))
    for figure in sorted((development_run / "figures").glob("*.png")):
        display(Image(filename=str(figure)))
else:
    print("No scientific results executed in implementation mode.")


### Optional: preserve the FAST-mode comparison

Run this cell after a FAST run, before approving a FULL run. The guard prevents accidentally overwriting the fast-mode table with full-mode results. To compare with a previously saved fast run, mount or restore its private CSV rather than relabeling the current run.

In [ ]:
if FAST_MODE and development_run is not None:
    fast_run = development_run
    fast_leaderboard = leaderboard_export.copy()

    fast_leaderboard.to_csv(
        "/kaggle/working/fast_leaderboard.csv",
        index=False
    )

    print("Fast mode results saved!")
    print("Fast run directory:", fast_run)
else:
    print('FAST backup skipped; this is not a FAST development run.')


## 8. Validation-only threshold sensitivity

Use the exact selected checkpoint and the original threshold-selection partition. Compare the Random Forest and the past-only personal prevalence baseline on the same rows. The historical run used 139 validation rows in this partition. This is a development estimate, not independent final evidence. The recall ≥60% display is exploratory, not a clinical requirement.

The following analysis does not retrain the model or change its frozen threshold.

In [ ]:
if development_run is not None and development_result["winner"]["model_name"] == "random_forest":
    import numpy as np
    import pandas as pd
    import joblib
    import matplotlib.pyplot as plt

    from IPython.display import display, FileLink
    from ml.cognitive.calibration import calibration_partitions
    from ml.cognitive.baselines import baseline_scores
    from ml.cognitive.evaluation import metrics

    # =====================================
    # 1. Load existing trained checkpoint
    # =====================================

    checkpoint = joblib.load(
        development_run / "models/selected_checkpoint.joblib"
    )

    model = checkpoint["model"]
    calibration = checkpoint["calibration"]

    # Validation data only
    bundle = load_and_validate(config)

    train_df = bundle.partition("train")
    val_df = bundle.partition("validation")

    # Use the original threshold-selection partition
    adequacy_df, threshold_df = calibration_partitions(val_df)

    print("Model:", model.model_name)
    print("Frozen threshold:", calibration.threshold)
    print("Threshold evaluation samples:", len(threshold_df))

    # =====================================
    # 2. Generate validation risk scores
    # =====================================

    rf_scores = calibration.apply(
        model.scores(threshold_df)
    )

    baseline_predictions, coverage = baseline_scores(
        train_df,
        threshold_df,
        history=bundle.history
    )

    personal_scores = baseline_predictions[
        "B2_PERSONAL_PREVALENCE"
    ]

    y_true = threshold_df[
        "target_elevated_fatigue"
    ].astype(int).to_numpy()

    # =====================================
    # 3. Threshold sensitivity analysis
    # =====================================

    thresholds = sorted(set(
        np.round(np.arange(0.10, 0.81, 0.01), 2).tolist()
        + [float(calibration.threshold), 0.5]
    ))

    rows = []

    for name, scores in [
        ("Random Forest E5", rf_scores),
        ("Personal Prevalence", personal_scores)
    ]:
        for threshold in thresholds:

            result = metrics(
                y_true,
                scores,
                threshold=threshold
            )

            tn, fp = result["confusion_matrix"][0]
            fn, tp = result["confusion_matrix"][1]

            precision = (
                tp / (tp + fp)
                if (tp + fp) > 0 else 0
            )

            rows.append({
                "model": name,
                "threshold": threshold,
                "n": len(y_true),
                "average_precision": result["average_precision"],
                "roc_auc": result["roc_auc"],
                "recall": result["recall"],
                "specificity": result["specificity"],
                "precision": precision,
                "macro_f1": result["macro_f1"],
                "balanced_accuracy": result["balanced_accuracy"],
                "true_positive": tp,
                "false_positive": fp,
                "false_negative": fn,
                "true_negative": tn
            })

    threshold_results = pd.DataFrame(rows)

    # =====================================
    # 4. Display selected thresholds
    # =====================================

    selected_thresholds = [
        0.20, 0.25, 0.30,
        0.35, 0.40, 0.45,
        0.49, 0.50, 0.55, 0.60
    ]

    display_columns = [
        "model",
        "threshold",
        "recall",
        "specificity",
        "precision",
        "macro_f1",
        "balanced_accuracy"
    ]

    display(
        threshold_results[
            threshold_results["threshold"].isin(
                selected_thresholds
            )
        ][display_columns]
        .style.format(precision=4)
    )

    # =====================================
    # 5. Best threshold by Macro F1
    # =====================================

    print("\nBEST MACRO F1 FOR EACH MODEL")

    best_f1 = (
        threshold_results
        .sort_values("macro_f1", ascending=False)
        .groupby("model", as_index=False)
        .head(1)
    )

    display(
        best_f1[display_columns]
        .style.format(precision=4)
    )

    # =====================================
    # 6. Recall-oriented candidates
    # =====================================

    print("\nCANDIDATES WITH RECALL >= 60%")

    recall_candidates = (
        threshold_results[
            threshold_results["recall"] >= 0.60
        ]
        .sort_values(
            ["specificity", "macro_f1"],
            ascending=False
        )
    )

    display(
        recall_candidates[display_columns]
        .head(15)
        .style.format(precision=4)
    )

    # =====================================
    # 7. Plot threshold trade-offs
    # =====================================

    for name in threshold_results["model"].unique():

        subset = threshold_results[
            threshold_results["model"] == name
        ]

        plt.figure(figsize=(10, 5))

        plt.plot(
            subset["threshold"],
            subset["recall"],
            label="Recall"
        )

        plt.plot(
            subset["threshold"],
            subset["specificity"],
            label="Specificity"
        )

        plt.plot(
            subset["threshold"],
            subset["macro_f1"],
            label="Macro F1"
        )

        plt.plot(
            subset["threshold"],
            subset["precision"],
            label="Precision"
        )

        plt.axvline(
            calibration.threshold,
            linestyle="--",
            color="black",
            label="RF frozen threshold"
        )

        plt.xlabel("Decision threshold")
        plt.ylabel("Metric")
        plt.title(name + " — Threshold Sensitivity")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()

    # =====================================
    # 8. Export downloadable results
    # =====================================

    output_file = "/kaggle/working/threshold_analysis.csv"

    threshold_results.to_csv(
        output_file,
        index=False
    )

    print("Saved:", output_file)

    display(
        FileLink("threshold_analysis.csv")
    )
else:
    print('Threshold analysis requires a completed Random Forest development run.')


## 9. Frozen model approval

The completed full run selected Random Forest E5 with threshold 0.49, native risk scores, and the train-only refit policy. This cell deliberately verifies that exact selection, the checkpoint SHA-256 and the configuration fingerprint. A different result from a new run requires a new documented ML lead decision; do not bypass the assertions.

Enable `APPROVE_FROZEN_EVALUATION` only after review of development results, and authorize the three protected evaluation scopes. The selected model is not refitted on the validation or test partitions. Calibration adequacy remains unapproved.

In [ ]:
if APPROVE_FROZEN_EVALUATION:
    from pathlib import Path
    from datetime import datetime, timezone
    from dataclasses import replace
    from ml.cognitive.approval import require_approval
    from ml.cognitive.config import file_hash
    import json

    # ==========================================
    # 1. Load the selected frozen configuration
    # ==========================================

    FROZEN_CONFIG = str(
        development_run / "configs/frozen_config.json"
    )

    freeze = json.loads(
        Path(FROZEN_CONFIG).read_text()
    )

    # Verify the selected model
    assert freeze["model_name"] == "random_forest"
    assert freeze["feature_set"] == "E5"
    assert abs(
        freeze["calibration"]["threshold"] - 0.49
    ) < 1e-9

    assert freeze["refit_policy"] == "train_only"
    assert freeze["calibration"]["probability_verified"] is False

    # Verify the original trained checkpoint
    checkpoint_path = (
        development_run
        / freeze["checkpoint_relative_path"]
    )

    assert file_hash(checkpoint_path) == freeze["checkpoint_sha256"]

    # Verify that configuration has not changed
    assert config.config_fingerprint == freeze["config_fingerprint"]

    # ==========================================
    # 2. Load previous ML lead approval
    # ==========================================

    previous_approval = json.loads(
        Path(config.approval_record).read_text()
    )

    # ==========================================
    # 3. Approve protected evaluation scopes
    # ==========================================

    frozen_approval = previous_approval.copy()

    frozen_approval.update({

        "approved": True,

        "approved_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "approved_freeze_fingerprint":
            freeze["freeze_fingerprint"],

        "authorized_scopes": [
            "final_chronological",
            "grouped_outer",
            "sensitivity_test"
        ],

        "calibration_adequacy_approved": False
    })

    # ==========================================
    # 4. Save a separate approval record
    # ==========================================

    frozen_approval_path = Path(
        "/kaggle/working/agent3_frozen_approval.json"
    )

    frozen_approval_path.write_text(
        json.dumps(frozen_approval, indent=2),
        encoding="utf-8"
    )

    # Bind the frozen configuration and approval
    APPROVAL_RECORD = str(frozen_approval_path)

    config = replace(
        config,
        approval_record=APPROVAL_RECORD,
        final_config=FROZEN_CONFIG,
        gpu_smoke=False
    )

    # ==========================================
    # 5. Verify approval using Agent 3's gate
    # ==========================================

    data = load_and_validate(config)

    for scope in [
        "final_chronological",
        "grouped_outer",
        "sensitivity_test"
    ]:
        require_approval(
            config,
            data,
            scope=scope,
            freeze=freeze
        )
        print(f"Approval verified: {scope}")

    # Verify all sensitivity scenario approvals
    for scenario in [
        "offset_plus1",
        "offset_plus2",
        "wellness_lag2"
    ]:
        scenario_data = load_and_validate(
            config, scenario
        )

        require_approval(
            config,
            scenario_data,
            scope="sensitivity_test",
            freeze=freeze
        )

        print(f"Scenario verified: {scenario}")

    # ==========================================
    # 6. Enable protected evaluation
    # ==========================================

    RUN_DEVELOPMENT = False

    RUN_FINAL_TEST = True
    RUN_GROUPED_OUTER = True
    RUN_SENSITIVITY_TEST = True

    # Export only after evaluation review
    RUN_EXPORT = False

    print("\nFROZEN MODEL APPROVAL SUCCESSFUL")
    print("Model:", freeze["model_name"])
    print("Feature set:", freeze["feature_set"])
    print("Threshold:", freeze["calibration"]["threshold"])
    print("Freeze fingerprint:", freeze["freeze_fingerprint"])
    print("Approval record:", APPROVAL_RECORD)
    print("Ready for protected evaluation.")
else:
    print('BLOCKED: explicit frozen-model approval required.')


## 10. Explicit protected chronological test

Requires a deliberately enabled flag, final_chronological authorization, exact fingerprint/config/code freeze and the selected checkpoint checksum. It loads the exact frozen train-fitted pipeline and calibration; no fitting, tuning or validation refit occurs here. Test outcomes are never used to revise the model.

In [ ]:
if RUN_FINAL_TEST:
    final_run, final_result = execute(replace(config, mode="final", run_final_test=True))
else:
    print("BLOCKED: protected final test disabled.")


## 11. Participant-independent outer evaluation — CatBoost bug fix

The original grouped run failed when CatBoost's `get_best_iteration()` returned `None` without an evaluation set. The corrected branch below retains the original nested GroupKFold procedure: it falls back to `tree_count_` for this case. The fix runs in an isolated writable copy with a distinct code fingerprint and separately approved grouped freeze. It does not modify the original Random Forest checkpoint, chronological freeze, test results or export configuration.

Enable `APPROVE_GROUPED_BUGFIX` explicitly after approving this code correction. Use the same source version of `ml.cognitive` as the historical frozen run; if the source already contains a different patch, do not bypass the fingerprint checks. This cell replaces the original failing grouped execution cell.

In [ ]:
if RUN_GROUPED_OUTER and APPROVE_GROUPED_BUGFIX:
    from pathlib import Path
    from datetime import datetime, timezone
    from dataclasses import replace, asdict
    from ml.cognitive.config import fingerprint, file_hash
    from ml.cognitive.experiment_logger import code_fingerprint

    import json
    import shutil
    import tempfile
    import subprocess
    import sys
    import textwrap

    # ==================================================
    # 1. ML lead approval for the GroupKFold bug fix
    # ==================================================

    assert APPROVE_GROUPED_BUGFIX is True, (
        "Review and approve the CatBoost bug fix first."
    )

    # Preserve the original frozen model
    original_freeze_path = Path(config.final_config)

    original_freeze = json.loads(
        original_freeze_path.read_text()
    )

    assert original_freeze["model_name"] == "random_forest"
    assert original_freeze["feature_set"] == "E5"
    assert original_freeze["calibration"]["threshold"] == 0.49
    assert original_freeze["code_fingerprint"] == code_fingerprint()

    # ==================================================
    # 2. Copy Agent 3 code into a writable directory
    # ==================================================

    patch_root = Path(tempfile.mkdtemp(
        prefix="agent3_grouped_fix_",
        dir="/kaggle/working"
    ))

    shutil.copytree(
        CODE_ROOT / "ml",
        patch_root / "ml",
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc")
    )

    models_path = patch_root / "ml/cognitive/models.py"

    source = models_path.read_text()

    old = (
        "elif name=='catboost':"
        "best=estimator.get_best_iteration()+1 "
        "if estimator.get_best_iteration()>=0 "
        "else estimator.tree_count_"
    )

    new = (
        "elif name=='catboost':\n"
        "                    iteration = estimator.get_best_iteration()\n"
        "                    best = iteration + 1 "
        "if iteration is not None and iteration >= 0 "
        "else estimator.tree_count_"
    )

    assert source.count(old) == 1, (
        "Expected CatBoost code not found."
    )

    patched_source = source.replace(old, new)

    # Verify Python syntax before saving
    compile(patched_source, str(models_path), "exec")

    models_path.write_text(patched_source)

    print("CatBoost bug fixed.")

    # ==================================================
    # 3. Create a separately frozen grouped procedure
    # ==================================================

    patched_code_fingerprint = fingerprint({
        p.name: file_hash(p)
        for p in sorted(
            (patch_root / "ml/cognitive").glob("*.py")
        )
    })

    grouped_freeze = original_freeze.copy()

    grouped_freeze["code_fingerprint"] = (
        patched_code_fingerprint
    )

    grouped_freeze["freeze_fingerprint"] = fingerprint({
        key: value
        for key, value in grouped_freeze.items()
        if key != "freeze_fingerprint"
    })

    grouped_freeze_path = (
        original_freeze_path.parent
        / f"{patch_root.name}_frozen.json"
    )

    grouped_freeze_path.write_text(
        json.dumps(grouped_freeze, indent=2)
    )

    # ==================================================
    # 4. Authorize only the patched GroupKFold procedure
    # ==================================================

    grouped_approval = json.loads(
        Path(config.approval_record).read_text()
    )

    grouped_approval.update({
        "approved": True,
        "approved_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "authorized_scopes": ["grouped_outer"],

        "approved_freeze_fingerprint":
            grouped_freeze["freeze_fingerprint"],

        "source_freeze_fingerprint":
            original_freeze["freeze_fingerprint"],

        "patch_description": (
            "Handle CatBoost get_best_iteration() "
            "returning None when no eval_set is supplied. "
            "Use actual tree_count_ in that case."
        ),

        "calibration_adequacy_approved": False
    })

    grouped_approval_path = (
        patch_root / "grouped_approval.json"
    )

    grouped_approval_path.write_text(
        json.dumps(grouped_approval, indent=2)
    )

    # ==================================================
    # 5. Configure grouped evaluation
    # ==================================================

    grouped_config = replace(
        config,
        mode="grouped",
        run_grouped_outer=True,
        run_final_test=False,
        run_sensitivity_test=False,
        run_export=False,
        gpu_smoke=False,
        resume_run=None,
        approval_record=str(grouped_approval_path),
        final_config=str(grouped_freeze_path)
    )

    assert (
        grouped_config.config_fingerprint
        == original_freeze["config_fingerprint"]
    )

    grouped_config_path = (
        patch_root / "grouped_config.json"
    )

    grouped_config_path.write_text(
        json.dumps(asdict(grouped_config), indent=2)
    )

    # ==================================================
    # 6. Execute in an isolated Python process
    # ==================================================

    result_path = patch_root / "grouped_result.json"

    runner = textwrap.dedent("""
    import json
    import sys
    from pathlib import Path

    from ml.cognitive.config import load_config
    from ml.cognitive.train import execute

    cfg = load_config(sys.argv[1])

    run, result = execute(cfg)

    Path(sys.argv[2]).write_text(
        json.dumps({
            "run": str(run),
            "result": result
        }, indent=2, default=str)
    )

    print("GROUPED EVALUATION COMPLETED")
    print(json.dumps(result, indent=2))
    """)

    print("\nStarting corrected GroupKFold evaluation...\n")

    process = subprocess.Popen(
        [
            sys.executable,
            "-u",
            "-c",
            runner,
            str(grouped_config_path),
            str(result_path)
        ],
        cwd=str(patch_root),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            "Grouped evaluation failed. "
            "Review the output above."
        )

    # ==================================================
    # 7. Load the corrected evaluation results
    # ==================================================

    grouped_output = json.loads(
        result_path.read_text()
    )

    grouped_run = Path(grouped_output["run"])
    grouped_result = grouped_output["result"]

    print("\nFINAL GROUPED RESULT")
    print(json.dumps(grouped_result, indent=2))

    print("\nResults directory:", grouped_run)
    print("Original Random Forest freeze preserved.")
else:
    print('BLOCKED: grouped evaluation or its bug-fix approval is disabled.')


## 12. Explicit protected sensitivity tests

Separate provided UTC+1, UTC+2 and wellness t−2 datasets/manifests; never mix scenarios or shift Fitbit timestamps. Frozen reference family/feature/iteration choices are refitted on each scenario's train rows; its lawful calibration/threshold protocol uses training/validation only. No sensitivity test outcome can revise the reference selection.

In [ ]:
if RUN_SENSITIVITY_TEST:
    sensitivity_run, sensitivity_result = execute(replace(config, mode="sensitivity-test", run_sensitivity_test=True))
else:
    print("BLOCKED: protected sensitivity test evaluation disabled.")


## 13. Evaluation review

Review chronological, grouped and sensitivity summaries before authorizing export. Grouped evaluation measures the independently selected procedure across participants, not the fixed chronological winner. Keep all individual-level predictions and metrics private. Do not change the selected model after looking at the protected test.

In [ ]:
for name in ("final_result", "grouped_result", "sensitivity_result"):
    value = globals().get(name)
    if isinstance(value, dict):
        print(f"\n{name.upper()}")
        print(json.dumps(value, indent=2, default=str))
    else:
        print(f"{name}: not available in this session")


## 14. CPU-compatible export and Agents 5/6 handoff

Authorize export only after reviewing completed protected evaluations. The original checkpoint and freeze are used, not the isolated grouped patch. Export records the feature order, model version, preprocessing fingerprints, target definition, threshold, calibration status, model card and CPU reload consistency. Backend/frontend implementation is out of scope.

Set `APPROVE_EXPORT = True` only after ML lead review; run the export approval cell, then the export execution cell.

In [ ]:
if APPROVE_EXPORT:
    from pathlib import Path
    from datetime import datetime, timezone
    from dataclasses import replace
    from ml.cognitive.approval import require_approval

    import json

    # Require the evaluation results to be available for the export review.
    assert isinstance(globals().get("final_result"), dict)
    assert final_result.get("status") == "protected_final_completed"
    assert isinstance(globals().get("grouped_result"), dict)
    assert isinstance(globals().get("sensitivity_result"), dict)
    assert sensitivity_result.get("status") == "sensitivity_completed"

    # Use the ORIGINAL frozen model,
    # not the patched GroupKFold configuration
    FROZEN_CONFIG = str(
        development_run / "configs/frozen_config.json"
    )

    freeze = json.loads(
        Path(FROZEN_CONFIG).read_text()
    )

    assert freeze["model_name"] == "random_forest"
    assert freeze["feature_set"] == "E5"
    assert freeze["calibration"]["threshold"] == 0.49

    # Load previous frozen approval
    approval = json.loads(
        Path(
            "/kaggle/working/agent3_frozen_approval.json"
        ).read_text()
    )

    # Authorize model export
    approval.update({
        "approved": True,
        "approved_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "authorized_scopes": ["export"],

        "approved_freeze_fingerprint":
            freeze["freeze_fingerprint"],

        "calibration_adequacy_approved": False
    })

    # Save separate export approval
    export_approval_path = Path(
        "/kaggle/working/agent3_export_approval.json"
    )

    export_approval_path.write_text(
        json.dumps(approval, indent=2)
    )

    APPROVAL_RECORD = str(export_approval_path)

    config = replace(
        config,
        mode="export",
        approval_record=APPROVAL_RECORD,
        final_config=FROZEN_CONFIG,
        run_export=True,
        run_final_test=False,
        run_grouped_outer=False,
        run_sensitivity_test=False,
        gpu_smoke=False,
        resume_run=None
    )

    # Validate export authorization
    data = load_and_validate(config)

    require_approval(
        config,
        data,
        scope="export",
        freeze=freeze
    )

    RUN_EXPORT = True

    print("EXPORT APPROVAL SUCCESSFUL")
    print("Model:", freeze["model_name"])
    print("Threshold:", freeze["calibration"]["threshold"])
    print("Ready for CPU-compatible export.")
else:
    print('BLOCKED: explicit export approval required.')


In [ ]:
if RUN_EXPORT:
    export_run, export_result = execute(replace(config, mode="export", run_export=True))
    print(json.dumps(export_result, indent=2))
else:
    print("BLOCKED: trained-model export disabled.")


## 15. Download private handoff artifacts

Download the model bundle and reports from `/kaggle/working` before closing the session. Preserve the original approval records and checkpoint privately. The model bundle is CPU-compatible and requires the matching `ml.cognitive` Python modules and dependencies. The report ZIP is for internal review; inspect its contents before sharing, particularly grouped and sensitivity reports. Do not publish participant-level records, fitted models, private dataset manifests or approval records to the public repository.

In [ ]:
if RUN_EXPORT and isinstance(globals().get('export_result'), dict):
    import shutil

    from pathlib import Path
    from IPython.display import display, FileLink

    model_dir = Path(
        export_result["artifact_path"]
    )

    assert export_result["status"] == "export_completed"
    assert export_result["cpu_reload_verified"] is True

    zip_path = shutil.make_archive(
        "/kaggle/working/agent3_final_model",
        "zip",
        root_dir=model_dir
    )

    print("Model package created:", zip_path)

    print("\nExported files:")

    for file in model_dir.iterdir():
        print(file.name)

    display(
        FileLink("agent3_final_model.zip")
    )
else:
    print('No authorized export artifact available.')


In [ ]:
if all(globals().get(name) is not None for name in ('development_run', 'final_run', 'grouped_run', 'sensitivity_run', 'final_result', 'grouped_result', 'sensitivity_result')):
    import json
    import zipfile
    from pathlib import Path
    from IPython.display import FileLink, display

    report_zip = Path(
        "/kaggle/working/agent3_final_reports.zip"
    )

    # Only selected aggregate reports
    run_files = {
        "development": (
            development_run,
            [
                "metrics/model_feature_results.csv",
                "metrics/primary_validation_baseline_metrics.csv",
                "metrics/runtime_summary.csv",
                "metrics/supplementary_results.csv",
                "metrics/sensitivity_results.csv",
                "metrics/calibration_metrics.json",
                "configs/frozen_config.json"
            ]
        ),

        "final_test": (
            final_run,
            [
                "metrics/primary_test_baseline_metrics.csv",
                "metrics/runtime_summary.csv"
            ]
        ),

        "grouped": (
            grouped_run,
            [
                "metrics/protected_grouped_fold_results.csv",
                "metrics/protected_grouped_pooled_metrics.json",
                "metrics/runtime_summary.csv"
            ]
        ),

        "sensitivity": (
            sensitivity_run,
            [
                "metrics/protected_sensitivity_results.csv",
                "metrics/protected_sensitivity_key_comparison.csv",
                "metrics/runtime_summary.csv"
            ]
        )
    }

    with zipfile.ZipFile(
        report_zip,
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:

        for stage, (run, filenames) in run_files.items():

            for filename in filenames:

                path = Path(run) / filename

                if path.exists():
                    z.write(
                        path,
                        arcname=f"{stage}/{path.name}"
                    )
                    print("Added:", stage, path.name)

        # Aggregate final test result
        z.writestr(
            "final_test/final_summary.json",
            json.dumps(final_result, indent=2)
        )

        for stage, result in [
            ("grouped", grouped_result),
            ("sensitivity", sensitivity_result),
        ]:
            z.writestr(
                f"{stage}/summary.json",
                json.dumps(result, indent=2, default=str)
            )

        # Additional threshold analysis
        for filename in [
            "threshold_analysis.csv",
            "fast_leaderboard.csv"
        ]:
            path = Path("/kaggle/working") / filename

            if path.exists():
                z.write(path, arcname=filename)

    print("\nReport package:", report_zip)

    display(FileLink("agent3_final_reports.zip"))
else:
    print('Reports require completed development and protected evaluations.')


## 16. Historical result summary — completed 15 November 2026

These aggregate values were retained from the submitted executed notebook. They document the completed run, not the output of a fresh execution of this cleaned copy.

| Item | Result |
|---|---|
| Preprocessing | pmdata-preprocessing-1.0.0; 28 upstream checks passed |
| Chronological split | 934 train / 287 validation / 228 test participant-days |
| Selected model | Random Forest E5, tuned |
| Features | 68 approved features |
| Hyperparameters | 422 trees; max depth 6; min samples leaf 3; max features 1.0; no class weighting |
| Frozen decision threshold | 0.49 (validation-only macro F1) |
| Calibration | Native risk score; probability verification false |
| Validation average precision | 0.5248 (287 rows) |
| Full development suite runtime | 188.09 seconds |
| Final test average precision | 0.3619 |
| Final test ROC-AUC | 0.6487 |
| Final test macro F1 | 0.6134 |
| Final test balanced accuracy | 0.6028 |
| Final test recall / specificity | 0.3061 / 0.8994 |
| Final test Brier score | 0.1834 |
| Protected test support | 228 rows, 49 positive, 179 negative, 13 participants |
| Sensitivity evaluation | Three scenarios completed |
| Grouped evaluation | Original no-eval-set CatBoost bug corrected in isolated run; detailed results saved separately |
| Model export | `export_completed`; CPU reload verified |

### Scientific limitations

The model predicts future elevated self-reported fatigue, not measured cognitive impairment or astronaut readiness. The selected score is not a validated probability. Recall remains limited; negative predictions are not evidence of safety. Test AP should be interpreted against class prevalence and strong personal-history baselines. PMData is a small terrestrial convenience sample; device timezone, ingestion timing and domain transfer remain unresolved. No clinical or operational use is supported.

### Privacy and reproducibility

Keep the data, run directories, predictions, individual metrics, trained artifacts and approval records private. Record exact versioned code, data, schema, split, environment, configuration and checkpoint fingerprints. A fresh run can differ from the historical run; it must not silently replace the approved frozen model. Review the license limitations before redistribution. The original executed notebook is a separate private record; this copy intentionally clears execution outputs.